In [ ]:
# CASMI 2026 — own pipeline: library + analog propagation + FP model + ranker (offline)
import glob, os, shutil, subprocess, sys, time
T0 = time.time()
tag = f"cp{sys.version_info.major}{sys.version_info.minor}"  # one wheel per Python version is shipped
whl = [w for w in glob.glob("/kaggle/input/**/rdkit*.whl", recursive=True) if f"-{tag}-" in w][:1]
print("wheels", whl)
if whl:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q", *whl], check=False)
import rdkit; print("rdkit", rdkit.__version__)
ART = os.path.dirname(glob.glob("/kaggle/input/**/casmi26_artifacts.txt", recursive=True)[0])
os.makedirs("/kaggle/working/code/casmi", exist_ok=True)  # writable copy: numba caches next to the code
for f in glob.glob(f"{ART}/code__*.py"):
    shutil.copy(f, "/kaggle/working/code/casmi/" + os.path.basename(f)[len("code__"):])
sys.path.insert(0, "/kaggle/working/code")
COMP = os.path.dirname(glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0])
print("artifacts", ART, "competition", COMP)

In [ ]:
import numpy as np, pandas as pd
from casmi import library as L
from casmi import pipeline as P
from casmi.search import Library, Pool
pool = Pool(ART)
lib = Library(L.load(f"{ART}/library.npz"), pool)
test = pd.read_parquet(f"{COMP}/test.parquet")
print(len(test), "spectra", test.molecule_id.nunique(), "molecules", f"{time.time() - T0:.0f}s")

In [ ]:
ranker = P.load_ranker(ART)
fpm = P.load_fp_models(ART)
print("artifact files:", sorted(os.listdir(ART)))
print("ranker:", "loaded" if ranker is not None else "NONE (heuristic)",
      "| fp models:", 0 if fpm is None else len(fpm["nets"]))
rows = P.run(test, pool, lib, ranker=ranker, fp_models=fpm, use_gen=True)
print(f"ranked {len(rows)} molecules, {time.time() - T0:.0f}s")

In [ ]:
from casmi.metric import candidate_key
sub_ids = pd.read_csv(f"{COMP}/sample_submission.csv").molecule_id
keycache = P.load_keycache(ART)
out = {}
for mid, smiles in rows:
    picked, seen = [], set()
    for s in smiles:
        k = keycache.get(s) or candidate_key(s)
        if k is None or k in seen:
            continue
        seen.add(k); picked.append(s)
        if len(picked) == 25:
            break
    out[mid] = picked
sub = pd.DataFrame({"molecule_id": sub_ids,
                    "smiles": [";".join(out.get(m) or ["C"]) for m in sub_ids]})
sub.to_csv("/kaggle/working/submission.csv", index=False)
n = sub.smiles.str.split(";").str.len()
print(sub.shape, "candidates/molecule min/median", n.min(), n.median(), f"{time.time() - T0:.0f}s"); sub.head()